# Lab 1: Environment Setup and Signal Acquisition

Welcome to the first software lab for this course! The purpose of this lab is to verify that your Python programming environment is configured correctly and to introduce the basic computational workflow that will be used throughout the semester. You will also complete a signal acquisition task based on simulated scalp EEG signals.

## 1. Setup

Before beginning an analysis, it is important to confirm that the computational environment contains all required packages and that the course-specific code can be imported successfully.

The installed Python and package versions will be printed beneath the cell. If the cell runs without errors, your Conda environment are likely configured correctly. If you encounter an import error, please first confirm that you launched `jupyter lab` from the correct Conda environment and that you are running the notebook from the intended project directory.

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)

## 2. EEG Signal Acquisition Practice

An electroencephalogram (EEG) is an non-invasive way to measure electrical activity in the brain. EEG uses small, metal discs called electrodes that attach to the scalp. In this section, we will explore electrophysiology signal acquisition using a simulated EEG dataset.

<p align="center">
  <img src="assets/1020.jpg" alt="Image" width="300">
</p>
image credit: https://docs.openbci.com/GettingStarted/Biosensing-Setups/EEGSetup/

### 2.1 Raw Signal Visualization

We will be looking at a synthetic 16-channel EEG dataset. Check the above image for the corresponding spatial positions of the 16 channels used.

In [ ]:
from src.lab.eeg_simulation import CHANNEL_NAMES, simulate_eeg

sample_rate_hz, time, clean_erp, recorded_eeg = simulate_eeg()
channel_index = {name: i for i, name in enumerate(CHANNEL_NAMES)}
print("trials, channels, samples:", recorded_eeg.shape)
print("channels:", CHANNEL_NAMES)

Physiological measurements such as EEG are commonly represented as multichannel time series. Before starting analysis, it's a good idea to inspect the raw data. Visualization can reveal differences in signal amplitude, temporal structure, noise level, channel quality, and condition-dependent patterns. These observations could be important in guiding later decisions.

In [ ]:
# TODO: Visualize one trial of raw C3 channel signal
c3 = channel_index["C3"]

C3_trial0 = ...
plt.plot(...)

plt.axvline(0, color="k", linestyle="--", linewidth=0.8)
plt.title('C3 trial 0')
plt.xlabel("Time from stimulus (ms)")
plt.ylabel("Voltage (µV)")

In [ ]:
# TODO: visualize the PSD for trial0 of channel C3, from 0Hz-100Hz
from src.lab.utils import compute_psd

f, psd = ...
plt.plot(...)
plt.xlim(0, 100)

plt.show()

**Question 1:** What should be the x and y axis label/unit for the above plot? 

**Answer:** 

### 2.2 Aliasing

Our EEG signal contains an Alpha component. Alpha rhythms are EEG oscillations in roughly the 8-12 Hz range, commonly observed during relaxed wakefulness, especially over central-posterior scalp regions. 

To reliably sample the alpha component, a sampling rate must exceed twice the highest signal frequency. Here, assume the signal has an 8 Hz alpha component. Compare sampling with 100 Hz and 10 Hz.

In [ ]:
# TODO: Calculate the frequency produced by Nyquist folding.
from src.lab.utils import sample_from_high_rate

def alias_frequency_hz(signal_frequency_hz, sample_rate_hz):
    return ...

time_100, sampled_100 = sample_from_high_rate(time, C3_trial0, 100)
time_10, sampled_10 = sample_from_high_rate(time, C3_trial0, 10)
print("8 Hz sampled at 10 Hz appears at:", alias_frequency_hz(8, 10), "Hz")

fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True, sharey=True)
for ax, sampled_time, sampled_signal, rate in zip(
    axs, (time_100, time_10), (sampled_100, sampled_10), (100, 10)
):
    ax.plot(time, C3_trial0, color="0.8", label="1 kHz source")
    ax.plot(sampled_time, sampled_signal, "o-", markersize=3, label=f"{rate} Hz samples")
    ax.set_ylabel("Voltage (µV)")
    ax.legend()
axs[-1].set_xlabel("Time (s)")
fig.tight_layout()

In [ ]:
# TODO: Visualize the psd of the 100Hz sampled signal.
f, psd = ...
plt.plot(...)
plt.title('PSD, 100Hz sampled')

In [ ]:
# TODO: Visualize the psd of the 10Hz sampled signal.
f, psd = ...
plt.plot(...)
plt.title('PSD, 10Hz sampled')

**Question 2:** At what frequency did the 8Hz alpha component show up when sampled at 10 Hz? What is the minimum sufficient sampling rate needed so that the 8Hz component is represented correctly in the sampled data?

**Answer:** 

**Question 3:** When the full C3 signal is downsampled to 100 Hz, does aliasing occur?

**Answer:** 

### 2.3 Quantization

Recall from the lecture that, for an ideal ADC with full-scale range $FSR$ and $N$ bits,

$$\Delta = \frac{FSR}{2^N}, \qquad \sigma_q = \frac{\Delta}{\sqrt{12}}.$$

The second expression is the standard deviation of ideal uniformly distributed quantization noise.

In [ ]:
# TODO: Quantize a signal, determine the empirical quantization-noise SD,
# and compare it with the theoretical value.

def quantize_signal(signal_uv, bits, full_scale_range_uv):
    n_levels = 2**bits
    delta_uv = full_scale_range_uv / n_levels
    lower_uv = -full_scale_range_uv / 2
    upper_uv = full_scale_range_uv / 2 - delta_uv

    # Empiricial quantization noise
    clipped = np.clip(signal_uv, lower_uv, upper_uv)
    codes = np.round((clipped - lower_uv) / delta_uv)
    reconstructed = codes * delta_uv + lower_uv
    quantization_error = ...
    sigma_q_empirical_uv = ...

    # Theoretical quantization noise
    sigma_q_theoretical_uv = ...

    return (
        reconstructed,
        quantization_error,
        delta_uv,
        sigma_q_empirical_uv,
        sigma_q_theoretical_uv
    )

# 8-bit quantization
(quantized_8bit,
    error_8bit,
    delta_8bit,
    sigma_q_empirical_8bit,
    sigma_q_theoretical_8bit
) = quantize_signal(C3_trial0, 8, 5000)

# 12-bit quantization
(quantized_12bit,
    error_12bit,
    delta_12bit,
    sigma_q_empirical_12bit,
    sigma_q_theoretical_12bit
) = quantize_signal(C3_trial0, 12, 5000)

In [ ]:
print(
    f"8-bit:"
    f"empirical σq = {sigma_q_empirical_8bit:.3f} µV, "
    f"theoretical σq = {sigma_q_theoretical_8bit:.3f} µV"
)

print(
    f"12-bit:"
    f"empirical σq = {sigma_q_empirical_12bit:.3f} µV, "
    f"theoretical σq = {sigma_q_theoretical_12bit:.3f} µV"
)

plt.figure(figsize=(9, 4))
plt.plot(time*1000, C3_trial0, label="Signal", linewidth=2, alpha=0.8)
plt.step(time*1000, quantized_8bit, where="mid", label="8-bit Quantization", alpha=0.5)
plt.step(time*1000, quantized_12bit, where="mid", label="12-bit Quantization", alpha=0.5)

plt.xlim(0, 500)
plt.xlabel("Time from stimulus (ms)")
plt.ylabel("Voltage (µV)")
plt.legend()
plt.show()

**Question 4:** What is the ratio between 4-bit ADC quantization noise and 8-bit ADC quantization noise? Can this ratio be calculated directly without explicitly calculating the quantization noise?

**Answer:** 

### 2.4 Amplification before digitization

Only for this section, assume that we can apply a gain before the adc by multiplying a signal with a constant.

Amplifying before the ADC reduces the quantization step expressed in input units, provided that the amplified signal does not clip.

In [ ]:
# TODO: Amplify by 24 fold before ADC, digitize, and convert back to input units.
def apply_gain_before_adc(signal_uv, gain, bits, adc_fsr_uv):
    signal, _, adc_delta_uv, _, _ = quantize_signal(signal_uv * gain, bits, adc_fsr_uv)
    delta_uv = adc_delta_uv / gain
    sigma_q_uv = ...
    gain_signal = ...
    return gain_signal, delta_uv, sigma_q_uv

no_gain_signal, _, _, no_gain_sigma, _ = quantize_signal(C3_trial0, 8, 5000)
gain_signal, _, gain_sigma = apply_gain_before_adc(C3_trial0, 24, 8, 5000)
print(f"No gain: σq={no_gain_sigma:.3f} µV")
print(f"Gain x24: σq={gain_sigma:.3f} µV")

**Question 5:** In this exercise, gain is multiplied in the digital computer. What assumption have we made when doing this? Where should gain be applied in the real world?

**Answer:** 

### 2.5 Theoretical SNR

**For this section only, assume we are working with `gain_signal` of channel C3, trial0 computed in section 2.4.**

Assume that the noise sources ar electrode noise, quantization noise, and line noise. We want to figure out the total noise and which component contributes the most. Recall from lecture that independent noise sources add in root-sum-square form:

$$V_{n,\mathrm{total}} = \sqrt{V_{n,\mathrm{electrode}}^2 + V_{n,\mathrm{line}}^2 + V_{n,\mathrm{quantization}}^2}$$

where electrode thermal noise RMS $V_{n,\mathrm{electrode}}$ can be estimated with the Johnson-Nyquist formula:

$$V_{n,\mathrm{electrode}} = \sqrt{4k_BTRB}$$

For 60 Hz line noise RMS $V_{n,\mathrm{line}}$ , use capacitive coupling model:

$$V(t) = V_{\text{peak}}\sin(2\pi f t)$$

$$i(t) = C\frac{dV}{dt}$$

$$i(t) = 2\pi f C V_{\text{peak}}\cos(2\pi f t)$$

$$I_{\text{line,rms}} = 2\pi f C V_{\text{main,rms}}$$

$$V_{n,\text{line}} = I_{\text{line,rms}}R_{\text{body}}$$


Use quantization noise you calculated in section 2.4 after applying gain:

$$V_{n,\mathrm{quantization}} = \sigma_q$$

Finally, use RMS voltages, SNR is then defined as:

$$SNR_{dB}=20\log_{10}(V_{signal}/V_{noise})$$

In [ ]:
# Constants and assumptions for theoretical SNR.
boltzmann_j_per_k = 1.380649e-23
temperature_k = 300
electrode_resistance_ohm = 1e6
bandwidth_hz = sample_rate_hz / 2

mains_rms_v = 120
mains_frequency_hz = 60
coupling_capacitance_f = 1.1e-15
body_resistance_ohm = 100e3

In [ ]:
# TODO: Combine independent theoretical noise components using RSS.
from src.lab.utils import rms

# 1. electrode noise
electrode_noise_rms_uv = ...

# 2. line noise
current_line_rms = ...
line_noise_rms_uv = ...

# 3. quantization noise
quantization_noise_rms_uv = ...

# 4. everything together
component_rms_uv = {
    "electrode Johnson-Nyquist": electrode_noise_rms_uv,
    "passive 60 Hz line coupling": line_noise_rms_uv,
    "quantization after gain": quantization_noise_rms_uv,
}
total_noise_rms_uv = ...
signal_rms_uv = ...
theoretical_snr_db = ...

for name, value in component_rms_uv.items():
    print(f"{name}: {value:.3f} µV RMS")
print(f"C3 gain_signal RMS: {signal_rms_uv:.2f} µV")
print(f"RSS total noise: {total_noise_rms_uv:.2f} µV")
print(f"Theoretical C3 SNR: {theoretical_snr_db:.2f} dB")

**Question 6:** Which component contributes the most noise according to the calculation? What can we do first to surpress that?

**Answer:** 

### 2.6 Empirical evoked-potential SNR

Real recordings do not expose their clean signal and noise components. Estimate empirical P300 SNR by averaging stimulus-aligned trials. Trial averaging suppresses uncorrelated activity while retaining the event-locked P300. 

Let's define signal to be the peak of 250ms ~ 350ms evoked potential:

$$\text{Signal}_{P300} = \max_{t \in [0.25,\,0.35]} \bar{x}(t)$$

and define noise as the prestimulus RMS of the averaged waveform (-200ms ~ 0ms):

$$\text{Noise} = \sqrt{\frac{1}{M}\sum_{t \in [-0.2,\,0]} \bar{x}(t)^2}$$

Thus:

$$\text{SNR}_{\mathrm{dB}} = 20\log_{10}\left(\frac{\text{Signal}_{P300}}{\text{Noise}}\right)$$

In [ ]:
# TODO: Average trials and estimate the P300 SNR from the evoked waveform.
def empirical_p300_snr_db(trials, time_s):
    evoked = np.mean(trials, axis=0)
    baseline_mask = (time_s >= -0.2) & (time_s < 0)
    p300_mask = (time_s >= 0.25) & (time_s <= 0.35)
    signal_uv = ...
    noise_uv = ...
    return ..., evoked

empirical_snr_db, raw_c3_evoked = empirical_p300_snr_db(recorded_eeg[:, c3], time)
print(f"Raw C3 empirical P300 SNR after {recorded_eeg.shape[0]}-trial averaging: {empirical_snr_db:.2f} dB")

plt.figure(figsize=(9, 4))
plt.plot(time * 1000, raw_c3_evoked, color="C0", linewidth=2, label="trial average")
plt.axvline(0, color="k", linestyle="--", linewidth=0.8)
plt.xlabel("Time from stimulus (ms)")
plt.ylabel("C3 voltage (µV)")
plt.legend()
plt.show()

**Question 7:** The above formulation assumed “signal” was the peak of the trial-averaged P300; What other methods could be used to extract the “signal” quantity of P300s? What are other ways you could define the “noise”?

**Answer:** 

### 2.7 CAR and Laplacian Referencing

Common-average referencing (CAR) subtracts the global channel mean.

$$ C3_{\mathrm{CAR}}(t) = C3(t) - \frac{1}{N}\sum_{i=1}^{N} V_i(t) $$

A local Laplacian uses neighboring sites, for C3:

$$C3_{Lap}=C3-(F3+P3+T3)/3$$


In [ ]:
# TODO: Compare raw, CAR, and AP local-Laplacian event-related signals.
car_eeg = ...
c3_laplacian = ...

raw_snr, raw_evoked = empirical_p300_snr_db(recorded_eeg[:, c3], time)
car_snr, car_evoked = empirical_p300_snr_db(car_eeg[:, c3], time)
lap_snr, c3_lap_evoked = empirical_p300_snr_db(c3_laplacian, time)
print(f"Raw C3 SNR: {raw_snr:.2f} dB")
print(f"CAR C3 SNR: {car_snr:.2f} dB")
print(f"C3 AP-Laplacian SNR: {lap_snr:.2f} dB")

fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True, sharey=True, dpi=150)
signals = (raw_evoked, car_evoked, c3_lap_evoked)
titles = ("Raw C3", "CAR C3", "Laplacian C3")
for ax, signal, title in zip(axs, signals, titles):
    ax.plot(time * 1000, signal)
    ax.axvline(0, color="k", linestyle="--", linewidth=0.8)
    ax.axvline(300, color="C3", linestyle=":", linewidth=1)
    ax.set_ylabel("µV")
    ax.set_title(title)
axs[-1].set_xlabel("Time from stimulus (ms)")
fig.suptitle("Trial-averaged ERP visibility after referencing", fontsize=20)
fig.tight_layout()

In [ ]:
# TODO: Compare trial-averaged PSDs after referencing.

trial_signals = {
    "Raw C3": recorded_eeg[:, c3],
    "CAR C3": car_eeg[:, c3],
    "Laplacian C3": c3_laplacian,
}

plt.figure(figsize=(8, 4), dpi=150)
for label, trials in trial_signals.items():
    trial_psds = []
    for trial in trials:
        f, psd = ...
        trial_psds.append(psd)
    mean_psd = ...
    plt.plot(f, mean_psd, label=label)

plt.xlim(0, 100)
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (µV²/Hz)")
plt.title("Trial-averaged C3 PSD after referencing")
plt.legend()
plt.tight_layout()

**Question 8:** Did the referencing methods improve the signal SNR? How are the two referencing methods working differently? 

**Answer:** 

**Question 9:** Which referencing method is better at preserving the amplitude of the ERP? What does this observation tell us about how the ERP is spatially distributed?

**Answer:** 

**Question 10:** Which referencing method is better at preserving the 8 Hz sinusoid? What does this observation tell us about how the 8Hz sinusoid is spatially distributed?

**Answer:** 

## Submission checklist

- Run all completed cells from top to bottom and answer the questions.
- Save the notebook.
- Submit the notebook itself to Gradescope.
- Export a PDF version and submit to Gradescope.